# Lesson 4 — Embeddings and Semantic Search

### AI Study Buddy · Lesson 4 of 8

This is the session that feels like magic, and then stops feeling like magic about forty
minutes in, which is the best thing that can happen to an idea.

The problem we ended on: *"what's the heaviest thing they can fly?"* and *"no payload
exceeds 2.8 kg in total mass"* are about the same thing and share **not one word**. Any
system that matches words will miss it.

So we need to compare text by **meaning** rather than by spelling. Which sounds impossible,
and turns out to be a solved problem you can use with one line of code.

## TODAY:

- **Part A:** turning words into numbers, and measuring meaning
- **Part B:** a real vector store over our documents
- **Part C:** searching it — and what happens when there's no answer

In [ ]:
# imports

import os
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
# Setup

load_dotenv(override=True)

DOCUMENTS_DIR = "documents"
DB_DIR = "vector_db"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMBEDDING_MODEL = "text-embedding-3-small"

## PART A: Turning words into numbers

An **embedding model** reads a piece of text and gives back a list of numbers — a
**vector** — that represents its meaning.

The crucial property: **texts that mean similar things get similar numbers.** Not similar
spelling. Similar *meaning*.

In [ ]:
# This model runs on OpenAI's servers, with the same key as the chat model. We send it
# text, it sends back numbers.
#
# It is not free, but it is close: $0.02 per million tokens. Every experiment in this
# lesson, for the whole room, costs less than a cent.

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

print("Embedding model ready")

In [ ]:
# Embed a single word and look at what comes out

vector = embeddings.embed_query("balloon")

print(f"Type:   {type(vector)}")
print(f"Length: {len(vector)} numbers")
print(f"First 8: {[round(v, 3) for v in vector[:8]]}")

### 1,536 numbers

Every piece of text — one word or five hundred — becomes exactly **1,536 numbers**.

Think of it as a position. Two numbers would be a position on a map: east-west and
north-south. Three would be a position in a room. This is a position in a space with 1,536
directions, and each direction represents some aspect of meaning that the model worked out
for itself during training. Nobody designed them, and mostly nobody can say what any
individual one means.

You can't picture 1,536 dimensions. You don't need to. You only need one thing from it:
**you can measure how far apart two positions are.**

In [ ]:
# Measuring distance. This is called cosine similarity - it asks
# "do these two vectors point in the same direction?"
#
# 1.0 = identical meaning, 0.0 = unrelated, negative = opposite.
# The maths is one line; what matters is that you can read the number.

def similarity(text_a, text_b):
    """How similar in meaning are two pieces of text? 1.0 is identical, 0.0 is unrelated."""
    a = np.array(embeddings.embed_query(text_a))
    b = np.array(embeddings.embed_query(text_b))
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
# Predict each of these before you run the cell. Write your guesses down.

pairs = [
    ("balloon", "balloon"),
    ("balloon", "airship"),
    ("balloon", "parachute"),
    ("balloon", "trombone"),
    ("car", "vehicle"),
    ("car", "automobile"),
    ("king", "queen"),
    ("hot", "cold"),
]

for a, b in pairs:
    print(f"{a:10} vs {b:12} → {similarity(a, b):.3f}")

### Look at `hot` vs `cold`

They're **opposites**, and the model scores them as highly similar. Is that wrong?

No — and it's worth a minute. The model has learned that "hot" and "cold" are *the same kind
of thing*: both are temperatures, both appear in the same sorts of sentences, both answer
the same sorts of questions. On the axis of "what topic is this about", they're nearly
identical.

Embeddings capture **topical similarity**, not agreement. That's usually exactly what you
want for search — someone asking about cold weather probably does want the paragraph about
temperature — but it's a real limitation, and it's the reason a retrieval system can
cheerfully hand back a paragraph that says the precise opposite of the truth.

### Now the pair that started all this

In [ ]:
# A question and its answer that share NOT ONE content word.
# Read them and check for yourself: radio/valleys, "get through"/"reach".

question = "What keeps working when the radio cannot get through?"
answer   = ("The satellite tracker is slower and more expensive, but it works in "
            "valleys where the LoRa signal does not reach.")
unrelated = "All Kestrel flights use lithium-based AA cells, never alkaline."

print(f"question vs the right answer : {similarity(question, answer):.3f}")
print(f"question vs an unrelated line: {similarity(question, unrelated):.3f}")

There it is. The question and its answer share **no words at all** — the question says
"radio" and "get through", the answer says "valleys" and "reach" — and the model still puts
them closer together. The unrelated sentence, which is also about the same project, scores
clearly lower.

A word search would score that question and that answer at **zero**. This is the thing
word searching can never do.

### Now try a different language

Some embedding models learn *meaning* across languages: they put a sentence near other
sentences that mean the same thing, whatever alphabet they happen to be written in. With
one of those, you could ask a question in Armenian and find a note written in English.

Is ours one of them? Predict before you run it.

In [ ]:
# Ask in Armenian. The notes are in English. Nothing is translated anywhere.
# Which line do you expect to score higher?

hy_question = "Ո՞ր թռիչքն է հասել ամենաբարձր կետին"          # "Which flight reached the highest point?"
en_answer   = "Flight 8 reached 34,600 m, the current programme record."
en_other    = "The box is wrapped in orange survival tape for visibility during recovery."

print(f"Armenian question vs the English answer   : {similarity(hy_question, en_answer):.3f}")
print(f"Armenian question vs an unrelated English line: {similarity(hy_question, en_other):.3f}")

Look carefully at those two numbers. The Armenian question scores the **unrelated**
sentence higher than the right answer. Cross-language search has failed — and nothing
printed an error to tell you.

This is a real limitation of `text-embedding-3-small`: it is excellent in English and much
weaker in Armenian. We measured it on the Kestrel files. Asked in English, it put the right
chunk in the top 4 for 7 questions out of 8. Asked in Armenian, it managed it for 1 out of 4.

So: notes in English work well. If your notes are in Armenian, check what retrieval actually
returns before you trust any answer — Lesson 7 shows how. Being able to *measure* a weakness
like this, instead of guessing, is the whole point of today.

<div style="border-left: 6px solid #f71; background: #fff7f0; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#f71; margin-top:0;">🚀 Extra challenge</h3>
<p style="color:#f71; margin-bottom:0;">
Cross-language matching is weak, not always wrong. Try to find an Armenian question whose
English answer scores <i>higher</i> than something irrelevant. Which kinds of question
work — short ones, ones with numbers or names in them? Finding the pattern tells you more
about the model than any single result.
</p>
</div>


<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Find a pair of texts where the model's answer <b>disagrees with you.</b><br/><br/>
Two things you consider closely related that it scores low, or two things you consider
unrelated that it scores high.<br/><br/>
They're not hard to find, and finding one is the point: this is a model with opinions
learned from what it read, not an oracle. Knowing where it's wrong is how you use it well.
</p>
</div>

In [ ]:
# Your experiments

print(similarity("", ""))

## PART B: A real vector store

We can embed a piece of text and compare two of them. Now scale it up: embed **every chunk**
of our documents, store the vectors, and search them.

The thing that does this is a **vector store** (or vector database). We'll use **Chroma**,
which runs locally and saves to a folder on your disk.

In [ ]:
# Load and split, exactly as in Lesson 3

def load_documents(folder):
    """Read every .md file in a folder and return them as LangChain Documents."""
    return [
        Document(page_content=path.read_text(encoding="utf-8"), metadata={"filename": path.name})
        for path in sorted(Path(folder).glob("**/*.md"))
    ]


documents = load_documents(DOCUMENTS_DIR)

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(documents)

print(f"{len(documents)} documents → {len(chunks)} chunks")

In [ ]:
# Build the store. This embeds every chunk, so it takes a few seconds -
# and it's the reason the real project will do this ONCE and save the result,
# rather than redoing it on every question.

# Start fresh. Re-running this cell would otherwise add a second copy of every chunk -
# and vectors from a different embedding model can't be mixed with these at all.
# (Ask Chroma to delete it, rather than deleting the folder: this notebook still has
# the database open, and pulling files out from under it breaks it.)
Chroma(persist_directory=DB_DIR, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR,
)

print(f"Stored {vectorstore._collection.count()} chunks in {DB_DIR}/")

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 The swap that makes this worth doing</h3>
<p style="color:#06c; margin-bottom:0;">
Look at what we handed <code>Chroma.from_documents</code>: some chunks, and an
<code>embedding</code> object. It has no idea <i>where</i> those embeddings are computed.<br/><br/>
So both halves are replaceable, and it's always a one-line change:<br/><br/>
<code>embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)               # OpenAI, what we use</code><br/>
<code># embeddings = OpenAIEmbeddings(model="text-embedding-3-large")     # sharper, ~6x the price</code><br/>
<code># embeddings = HuggingFaceEndpointEmbeddings(model="...")           # a different company</code><br/><br/>
That last line is worth a moment. Our answering model and our search model both come from
OpenAI today — but they don't have to. If a different company made an embedding model that
was much better at Armenian, you could switch to it without touching the answering half at
all.<br/><br/>
And the store itself: Chroma on your laptop today; a hosted database like Pinecone or
pgvector when you have ten million chunks and a company. Same
<code>.similarity_search()</code>, same code around it.<br/><br/>
That only works because the two halves aren't tangled together:
<code>retriever.py</code> and the model never touch each other. Building it that way
wasn't caution for its own sake — this is the situation it was for.
</p>
</div>


## PART C: Searching

In [ ]:
# Ask for the 3 chunks closest in meaning to the question.
# Note: there is no language model involved here at all. This is pure search.

results = vectorstore.similarity_search("What is the heaviest payload allowed?", k=3)

for i, doc in enumerate(results, start=1):
    print(f"--- Result {i}  (from {doc.metadata['filename']}) ---")
    print(doc.page_content[:250])
    print()

### Look carefully at what came back

The 2.8 kg rule is in there — but depending on how your chunks fell, it may not be the
**first** result. On our documents it typically comes back second or third, behind a chunk
about how a flight works.

Two things worth taking from that:

1. **It found it at all**, using the word "heaviest", which appears nowhere in any document.
   Word matching had no chance. This did.
2. **Being first isn't the job.** This is why we ask for `k=3` rather than `k=1` — we're not
   trying to pick *the* answer, we're trying to hand the model a small set of pieces that
   probably contains it. Choosing between them is the model's job, not the search's.

Run it again with `k=1` and see how much more fragile it gets.

In [ ]:
# The scores, so we can see how confident it is.
# For Chroma these are DISTANCES: lower is closer. (Some stores report it the other
# way round - always check, and never assume which direction "better" is.)

for doc, score in vectorstore.similarity_search_with_score("Why are there two trackers?", k=4):
    preview = doc.page_content[:70].replace("\n", " ")
    print(f"{score:.3f}  {doc.metadata['filename']:15} {preview}...")

### The question with no answer

This is the important experiment of the session. Search for something that **isn't in the
documents at all**.

In [ ]:
# Nothing in our Kestrel documents is about football.

for doc, score in vectorstore.similarity_search_with_score("Who won the 2018 World Cup?", k=3):
    preview = doc.page_content[:70].replace("\n", " ")
    print(f"{score:.3f}  {doc.metadata['filename']:15} {preview}...")

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🛑 Read this twice — it matters</h3>
<p style="color:#900; margin-bottom:0;">
It <b>still returned three chunks.</b><br/><br/>
A vector store <i>always</i> gives you the k nearest chunks. It has no concept of "nothing
here is relevant" — it just hands you the closest things it has, however far away they are.
The scores are worse, but it doesn't refuse.<br/><br/>
So if we simply paste whatever comes back into a prompt and ask for an answer, the model
will get three irrelevant paragraphs about weather balloons and a question about football,
and it will produce <i>something</i>.<br/><br/>
<b>That means retrieval alone does not prevent hallucination.</b> Next session we add the
other half: a system prompt that tells the model it is allowed — required — to say "that
isn't in the documents".
</p>
</div>

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Build a vector store over <b>your own</b> documents and search it.<br/><br/>
Then find these three cases and write down the scores for each:<br/>
1. A question it answers <b>well</b> — the right chunk comes back first<br/>
2. A question where the right chunk comes back, but <b>not first</b><br/>
3. A question about something not in your documents at all<br/><br/>
Compare the scores across all three. Is there a number where you'd draw the line between
"found it" and "found nothing"? Is that line in the same place for everyone in the room?
</p>
</div>

In [ ]:
# Your store. Using a different folder so it doesn't collide with the Kestrel one.

my_documents = load_documents("documents")
my_chunks = splitter.split_documents(my_documents)
Chroma(persist_directory="my_vector_db", embedding_function=embeddings).delete_collection()  # start fresh
my_store = Chroma.from_documents(my_chunks, embedding=embeddings, persist_directory="my_vector_db")

print(f"Stored {my_store._collection.count()} chunks")

In [ ]:
# Your three test questions

for question in ["", "", ""]:
    if not question:
        continue
    print(f"Q: {question}")
    for doc, score in my_store.similarity_search_with_score(question, k=3):
        preview = doc.page_content[:60].replace("\n", " ")
        print(f"   {score:.3f}  {doc.metadata['filename']:15} {preview}...")
    print()

<div style="border-left: 6px solid #f71; background: #fff7f0; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#f71; margin-top:0;">🚀 Optional: see the space</h3>
<p style="color:#f71; margin-bottom:0;">
1,536 dimensions can be squashed down to 2 so we can look at them. The technique is called
t-SNE, and the squashing loses a lot — but chunks that were close together mostly stay
close together, so the picture is meaningful.<br/><br/>
Each dot is a chunk, coloured by which file it came from. If the colours form clusters,
the embedding model has worked out which chunks are about the same topic <b>without ever
being told the filenames.</b>
</p>
</div>

In [ ]:
# Optional visualisation - skip this if you're short on time.

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

stored = vectorstore._collection.get(include=["embeddings", "metadatas"])
vectors = np.array(stored["embeddings"])
filenames = [m["filename"] for m in stored["metadatas"]]

# perplexity must be smaller than the number of points
flat = TSNE(n_components=2, random_state=42, perplexity=min(15, len(vectors) - 1)).fit_transform(vectors)

plt.figure(figsize=(9, 7))
for name in sorted(set(filenames)):
    picks = [i for i, f in enumerate(filenames) if f == name]
    plt.scatter(flat[picks, 0], flat[picks, 1], label=name, s=60, alpha=0.75)

plt.legend()
plt.title("Our chunks, squashed from 1,536 dimensions down to 2")
plt.show()

## Where we got to

- An embedding turns any text into 1,536 numbers representing its meaning
- Similar meaning → similar numbers, **regardless of the words used**. That's what
  word-matching could never do
- Embeddings capture *topical* similarity, not agreement — "hot" and "cold" score high
- Our model is strong in English and weak in Armenian — measure before you trust it
- A vector store holds every chunk's vector and finds the nearest ones to a question
- Both the embedding model and the store are swappable one-line changes
- **A vector store always returns k results, even when nothing is relevant** — so retrieval
  alone is not enough

## Next lesson

Everything connects. Retrieve the relevant chunks, put them in the system prompt from
Lesson 2, and ask the model. Plus the sentence that turns Lesson 2's confident liar into
something that admits when it doesn't know.